In [1]:
from config_info import APIS
from pprint import pprint

from scrapers.arxiv import parser_arxiv
from scrapers.hal import parser_hal
from scrapers.basic_fetching import fetch_raw

# Basic Fetch

# Semantic Scholar

In [ ]:
def semantic_fetch(results):
    for result in results["data"]:
        print(result)
        # query = APIS["Semantic Scholar"]["paper_url"].format(paper_id=result["paperId"])
        # res = fetch_raw(query)
        # print(res)
        # print(query)

# Main 

In [ ]:
# from models.postgres.corpus_schema import metadata, document_table
# from config.db_engine import get_db_engine
# metadata.create_all(get_db_engine())

In [4]:
from database.postgres.crud import upsert_data
from models.postgres.corpus_schema import document_table 
from config.db_engine import get_db_engine
from processing.cleaning_data import normalize_data

arxiv_mapping = {
    "published_at": "published",
}
raw_result = fetch_raw(APIS["arXiv"]["api_url"].format(query="Natural Language Processing",quantity='10'))
result_arxiv =  parser_arxiv(raw_result)
clean_arxiv_data = normalize_data(
    raw_data=result_arxiv, 
    source_name="arXiv",
    date_columns=["published_at"],
    columns_drop=["updated"]
    )


# hal_mapping = { "published" : "published_at"}
# clean_hal_data = normalize_data(
#     raw_data=result_hal,
#     source_name="Hal",
#     column_mapping=hal_mapping,
#     date_columns=["published"]
# )

# clean_pubmed_data = normalize_data(
#     raw_data= pubmed_list,
#     source_name="Pubmed",
#     date_columns=["published"]
# )
engine = get_db_engine()
upsert_data(clean_arxiv_data, ['id'], document_table, engine)


HTTPError: 429 Client Error: Unknown Error for url: https://export.arxiv.org/api/query?search_query=all:Natural%20Language%20Processing&start=0&max_results=10

In [ ]:
from storage.pdf_downloader import run_pdf_pipeline

run_pdf_pipeline(engine,document_table)